# AsyncFlow Live Telemetry Dashboard

Real-time visualization of RHAPSODY telemetry events as workflows execute.

Uses `telemetry.subscribe()` to receive every event as it fires and updates
four live plots via plotly `FigureWidget` (in-place mutations — no re-renders):

| Plot | Data source |
|---|---|
| Live counters | TaskSubmitted / TaskStarted / TaskCompleted / TaskFailed |
| Task stream (Gantt) | TaskStarted + TaskCompleted pairs |
| Concurrency curve | TaskStarted (+1) / TaskCompleted (-1) step chart |
| Resource utilization | ResourceUpdate (cpu_percent, memory_percent) |

**Requirements:** `pip install plotly ipywidgets`  
Run `jupyter nbextension enable --py widgetsnbextension` if widgets don't render.

In [1]:
import asyncio
import time
from collections import defaultdict, deque
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

from radical.asyncflow import WorkflowEngine
from rhapsody.backends import ConcurrentExecutionBackend
from rhapsody.telemetry.events import BaseEvent

kj/filesystem-disk-unix.c++:1734: warning: PWD environment variable doesn't match current directory; pwd = /home/aymen/RADICAL/RHAPSODY-BUGS


## Dashboard class

Subscribes to the telemetry bus and owns all live widgets.
Call `dashboard.display()` to render, then `await dashboard.start_refresh()`
to begin polling (0.5 s refresh rate by default).

In [2]:
class LiveTelemetryDashboard:
    """Real-time telemetry dashboard for AsyncFlow workflows.

    Subscribes to a TelemetryManager and renders four live plots
    inside a Jupyter notebook using plotly FigureWidget.

    Usage::

        dashboard = LiveTelemetryDashboard(telemetry)
        dashboard.display()                    # render widgets
        asyncio.ensure_future(dashboard.start_refresh())  # begin live updates
        # ... run workflows ...
        dashboard.stop()
    """

    TASK_COLORS = [
        "#2196f3", "#4caf50", "#ff9800", "#e91e63",
        "#9c27b0", "#00bcd4", "#8bc34a", "#ff5722",
    ]

    def __init__(self, telemetry, refresh_interval: float = 0.5, max_gantt_rows: int = 60):
        self._telemetry = telemetry
        self._refresh_interval = refresh_interval
        self._max_gantt_rows = max_gantt_rows
        self._running = False
        self._t0: float | None = None

        # ── shared state (written by subscriber callback, read by refresh loop) ──
        self._counters = {"submitted": 0, "started": 0, "completed": 0, "failed": 0}
        self._running_count = 0

        # concurrency step series: (t_ms, delta)
        self._conc_events: list[tuple[float, int]] = []

        # resource time series (bounded)
        self._res_t:   deque = deque(maxlen=400)
        self._res_cpu: deque = deque(maxlen=400)
        self._res_mem: deque = deque(maxlen=400)

        # task stream: completed tasks only (need both start and end)
        self._task_starts: dict[str, float] = {}   # task_id → start_ms
        self._completed_tasks: list[dict] = []     # all completed tasks
        self._gantt_rendered_count: int = 0        # how many have been drawn so far
        self._gantt_legend_names: set[str] = set() # names already shown in legend
        self._name_colors: dict[str, str] = {}     # name → color
        self._color_idx = 0

        # throughput: rolling completion timestamps
        self._completion_times: deque = deque(maxlen=1000)

        self._build_widgets()
        telemetry.subscribe(self._on_event)

    # ── event callback ────────────────────────────────────────────────────────

    def _on_event(self, event: BaseEvent) -> None:
        now_ms = self._rel(event.event_time)
        et = event.event_type

        if et == "TaskSubmitted":
            self._counters["submitted"] += 1

        elif et == "TaskStarted":
            self._counters["started"] += 1
            self._running_count += 1
            self._task_starts[event.task_id] = now_ms
            self._conc_events.append((now_ms, +1))

        elif et == "TaskCompleted":
            self._counters["completed"] += 1
            self._running_count = max(0, self._running_count - 1)
            self._conc_events.append((now_ms, -1))
            self._completion_times.append(now_ms)
            start_ms = self._task_starts.pop(event.task_id, now_ms)
            name = event.attributes.get("executable", "task")
            self._completed_tasks.append({
                "name": name, "start_ms": start_ms,
                "end_ms": now_ms, "status": "ok",
            })
            self._ensure_color(name)

        elif et == "TaskFailed":
            self._counters["failed"] += 1
            self._running_count = max(0, self._running_count - 1)
            self._conc_events.append((now_ms, -1))
            start_ms = self._task_starts.pop(event.task_id, now_ms)
            name = event.attributes.get("executable", "task")
            self._completed_tasks.append({
                "name": name, "start_ms": start_ms,
                "end_ms": now_ms, "status": "failed",
            })
            self._ensure_color(name)

        elif et == "ResourceUpdate" and getattr(event, "resource_scope", "") == "per_node":
            if event.cpu_percent is not None:
                self._res_t.append(now_ms)
                self._res_cpu.append(event.cpu_percent)
                self._res_mem.append(event.memory_percent or 0.0)

    def _rel(self, t: float) -> float:
        """Wall-clock time → ms relative to first event."""
        if self._t0 is None:
            self._t0 = t
        return (t - self._t0) * 1000.0

    def _ensure_color(self, name: str) -> None:
        if name not in self._name_colors:
            self._name_colors[name] = self.TASK_COLORS[self._color_idx % len(self.TASK_COLORS)]
            self._color_idx += 1

    # ── widget construction ───────────────────────────────────────────────────

    def _build_widgets(self) -> None:
        # Counter badges
        self._w_submitted  = widgets.HTML(self._badge("Submitted",  0, "#2196f3"))
        self._w_started    = widgets.HTML(self._badge("Running",    0, "#ff9800"))
        self._w_completed  = widgets.HTML(self._badge("Completed",  0, "#4caf50"))
        self._w_failed     = widgets.HTML(self._badge("Failed",     0, "#f44336"))
        self._w_throughput = widgets.HTML(self._badge("tasks/s",    0, "#9c27b0"))
        self._counter_row  = widgets.HBox(
            [self._w_submitted, self._w_started, self._w_completed,
             self._w_failed, self._w_throughput],
            layout=widgets.Layout(margin="8px 0"),
        )

        # Task stream (Gantt).
        # IMPORTANT: FigureWidget.data cannot be replaced with new trace objects —
        # it only accepts permutations/subsets of already-registered traces.
        # Strategy: grow with add_trace(), never remove. Slide the y-axis window
        # instead so only the last _max_gantt_rows bars are in the viewport.
        self._fig_gantt = go.FigureWidget(
            layout=go.Layout(
                title="Task Stream  (each bar = one completed task)",
                xaxis_title="Time (ms from session start)",
                yaxis=dict(visible=False),
                height=280,
                margin=dict(l=80, r=20, t=40, b=40),
                showlegend=True,
                legend=dict(orientation="h", y=-0.25),
                plot_bgcolor="#fafafa",
                barmode="overlay",
            )
        )

        # Concurrency + resource — two y-axes, 3 empty traces pre-registered
        # so _update_conc can always index fig.data[0..2] directly.
        self._fig_conc = go.FigureWidget(
            data=[
                go.Scatter(x=[], y=[], mode="lines", fill="tozeroy",
                           line=dict(color="#2196f3", width=2),
                           fillcolor="rgba(33,150,243,0.18)",
                           name="Tasks running"),
                go.Scatter(x=[], y=[], mode="lines+markers",
                           line=dict(color="#f44336", width=1.5),
                           marker=dict(size=4),
                           name="CPU %", yaxis="y2"),
                go.Scatter(x=[], y=[], mode="lines+markers",
                           line=dict(color="#ff9800", width=1.5, dash="dot"),
                           marker=dict(size=4, symbol="square"),
                           name="Memory %", yaxis="y2"),
            ],
            layout=go.Layout(
                title="Concurrency  +  Node Resources",
                xaxis_title="Time (ms)",
                yaxis=dict(title="Tasks running", color="#2196f3"),
                yaxis2=dict(title="Utilization (%)", overlaying="y",
                            side="right", range=[0, 105], color="#b71c1c"),
                height=260,
                margin=dict(l=80, r=80, t=40, b=40),
                legend=dict(orientation="h", y=-0.28),
                plot_bgcolor="#fafafa",
            )
        )

    @staticmethod
    def _badge(label: str, value, color: str) -> str:
        return (
            f'<div style="display:inline-block;background:{color};color:#fff;'
            f'border-radius:6px;padding:6px 14px;margin:0 6px;font-family:monospace;">'
            f'<span style="font-size:1.5em;font-weight:bold;">{value}</span>'
            f'<br><span style="font-size:0.75em;opacity:0.9;">{label}</span></div>'
        )

    # ── rendering ─────────────────────────────────────────────────────────────

    def display(self) -> None:
        """Render the dashboard in the current cell output."""
        display(self._counter_row, self._fig_gantt, self._fig_conc)

    # ── refresh loop ──────────────────────────────────────────────────────────

    async def start_refresh(self) -> None:
        """Coroutine: poll shared state and update widgets every refresh_interval.

        Schedule with asyncio.ensure_future() so it runs alongside the workflow.
        Exceptions inside _refresh() are caught and printed — never kill the loop.
        """
        self._running = True
        while self._running:
            try:
                self._refresh()
            except Exception as exc:  # noqa: BLE001
                import traceback
                print(f"[LiveDashboard] refresh error: {exc}")
                traceback.print_exc()
            await asyncio.sleep(self._refresh_interval)
        self._refresh()  # final update after stop()

    def stop(self) -> None:
        """Stop the refresh loop after the current sleep."""
        self._running = False

    def _refresh(self) -> None:
        """Push current shared state into all FigureWidgets."""
        self._update_counters()
        self._update_gantt()
        self._update_conc()

    def _update_counters(self) -> None:
        now_ms = (time.time() - self._t0) * 1000 if self._t0 else 0
        window = 3000  # ms — 3-second rolling window for throughput
        recent = sum(1 for t in self._completion_times if now_ms - t < window)
        tps = round(recent / (window / 1000), 1)
        self._w_submitted.value  = self._badge("Submitted",  self._counters["submitted"],  "#2196f3")
        self._w_started.value    = self._badge("Running",    self._running_count,           "#ff9800")
        self._w_completed.value  = self._badge("Completed",  self._counters["completed"],  "#4caf50")
        self._w_failed.value     = self._badge("Failed",     self._counters["failed"],     "#f44336")
        self._w_throughput.value = self._badge("tasks/s",    tps,                          "#9c27b0")

    def _update_gantt(self) -> None:
        """Incrementally add new task bars; slide y-axis window to show last N.

        Never removes traces — only grows via add_trace(). The y-axis window
        scrolls upward so only _max_gantt_rows bars remain in the viewport;
        older bars are out of view but still in the figure.

        Root cause of the original crash: FigureWidget.data = new_traces fails
        when new_traces contains trace objects not already registered with the
        figure. add_trace() is the only safe way to introduce new traces.
        The trim approach (fig.data = fig.data[-N:]) looked valid but caused
        the y-axis range [−0.5, N−0.5] to not match the kept traces whose
        y-positions were e.g. 40..99 — pushing all bars out of the viewport.
        """
        new_tasks = self._completed_tasks[self._gantt_rendered_count:]
        if not new_tasks:
            return

        # y-positions are 0-indexed and monotonically increase.
        y_base = self._gantt_rendered_count

        for i, t in enumerate(new_tasks):
            name = t["name"]
            color = self._name_colors.get(name, "#888")
            bar_color = "#f44336" if t["status"] == "failed" else color
            show_legend = name not in self._gantt_legend_names
            if show_legend:
                self._gantt_legend_names.add(name)

            self._fig_gantt.add_trace(
                go.Bar(
                    x=[t["end_ms"] - t["start_ms"]],
                    y=[y_base + i],
                    base=[t["start_ms"]],
                    orientation="h",
                    width=0.6,
                    marker_color=bar_color,
                    showlegend=show_legend,
                    name=name,
                    legendgroup=name,
                    hovertemplate=(
                        f"<b>{name}</b><br>"
                        "start: %{base:.0f} ms<br>"
                        "dur: %{x:.0f} ms<extra></extra>"
                    ),
                )
            )

        self._gantt_rendered_count += len(new_tasks)

        # Slide the y-axis window: always show the last _max_gantt_rows bars.
        # y-positions run 0..total-1; window = [total-N-0.5, total-0.5].
        total = self._gantt_rendered_count
        y_max = total - 0.5
        y_min = max(-0.5, total - self._max_gantt_rows - 0.5)
        with self._fig_gantt.batch_update():
            self._fig_gantt.layout.yaxis.range = [y_min, y_max]

    def _update_conc(self) -> None:
        # Sort by (time, -delta) so starts (+1) always precede completions (-1)
        # at equal timestamps — prevents running count from going negative.
        events = sorted(self._conc_events, key=lambda e: (e[0], -e[1]))
        t_vals, c_vals, running = [], [], 0
        for t_ms, delta in events:
            t_vals += [t_ms, t_ms]
            c_vals += [running, running + delta]
            running += delta

        r_t   = list(self._res_t)
        r_cpu = list(self._res_cpu)
        r_mem = list(self._res_mem)

        with self._fig_conc.batch_update():
            self._fig_conc.data[0].x = t_vals
            self._fig_conc.data[0].y = c_vals
            self._fig_conc.data[1].x = r_t
            self._fig_conc.data[1].y = r_cpu
            self._fig_conc.data[2].x = r_t
            self._fig_conc.data[2].y = r_mem

print("LiveTelemetryDashboard defined.")

LiveTelemetryDashboard defined.


## Workflow definition

Same 5-task DAG as example 08. Swap backend at the top of `main()` as needed.

In [3]:
async def main(n_workflows: int = 20):
    # ── backend + engine ───────────────────────────────────────────────────
    backend  = await ConcurrentExecutionBackend(ThreadPoolExecutor())
    flow     = await WorkflowEngine.create(backend)
    telemetry = await flow.start_telemetry(
        resource_poll_interval=0.1,   # 200 ms — fast enough for live view
    )

    # ── dashboard ──────────────────────────────────────────────────────────
    dashboard = LiveTelemetryDashboard(telemetry, refresh_interval=0.1)
    dashboard.display()

    # Start refresh loop as a background task — runs alongside the workflow.
    refresh_task = asyncio.ensure_future(dashboard.start_refresh())

    # ── task definitions ───────────────────────────────────────────────────

    @flow.function_task
    async def task_load(wf_id: int):
        await asyncio.sleep(0.05)
        return {"wf": wf_id, "rows": list(range(100))}

    @flow.function_task
    async def task_fetch(wf_id: int):
        await asyncio.sleep(0.03)
        return {"wf": wf_id, "external": list(range(50))}

    @flow.function_task
    async def task_clean(loaded: dict):
        await asyncio.sleep(0.04)
        return {**loaded, "rows": [r for r in loaded["rows"] if r % 2 == 0]}

    @flow.function_task
    async def task_join(cleaned: dict, fetched: dict):
        await asyncio.sleep(0.06)
        return {**cleaned, "merged": cleaned["rows"] + fetched["external"]}

    @flow.function_task
    async def task_report(cleaned: dict, joined: dict):
        await asyncio.sleep(0.02)
        return {
            "wf": cleaned["wf"],
            "rows": len(joined["merged"]),
            "ts": time.time(),
        }

    # ── DAG runner ─────────────────────────────────────────────────────────
    async def run_workflow(wf_id: int):
        loaded  = task_load(wf_id)
        fetched = task_fetch(wf_id)
        cleaned = task_clean(loaded)
        joined  = task_join(cleaned, fetched)
        return await task_report(cleaned, joined)

    # ── run ────────────────────────────────────────────────────────────────
    t0 = time.time()
    results = await asyncio.gather(*(run_workflow(i) for i in range(1, n_workflows + 1)))
    elapsed = time.time() - t0

    await flow.shutdown()
    await telemetry.stop()
    dashboard.stop()
    await asyncio.sleep(0.6)  # let the final refresh fire

    # ── summary ────────────────────────────────────────────────────────────
    s = telemetry.summary()
    print(f"\n{n_workflows} workflows in {elapsed*1000:.0f} ms")
    print(f"Tasks: {s['tasks']}")
    if s.get("duration"):
        d = s["duration"]
        print(f"Mean task time: {d['mean_seconds']*1000:.1f} ms  "
              f"Max: {d['max_seconds']*1000:.1f} ms")
    return results

## Run

The dashboard renders immediately below this cell.
Plots update every ~400 ms while workflows execute.

In [ ]:
results = await main(n_workflows=200)

FigureWidget({
    'data': [],
    'layout': {'barmode': 'overlay',
               'height': 280,
               'legend': {'orientation': 'h', 'y': -0.25},
               'margin': {'b': 40, 'l': 80, 'r': 20, 't': 40},
               'plot_bgcolor': '#fafafa',
               'showlegend': True,
               'template': '...',
               'title': {'text': 'Task Stream  (each bar = one completed task)'},
               'xaxis': {'title': {'text': 'Time (ms from session start)'}},
               'yaxis': {'visible': False}}
})

FigureWidget({
    'data': [{'fill': 'tozeroy',
              'fillcolor': 'rgba(33,150,243,0.18)',
              'line': {'color': '#2196f3', 'width': 2},
              'mode': 'lines',
              'name': 'Tasks running',
              'type': 'scatter',
              'uid': 'a7f64004-9c53-44d4-85be-a940540ac10d',
              'x': [],
              'y': []},
             {'line': {'color': '#f44336', 'width': 1.5},
              'marker': {'size': 4},
              'mode': 'lines+markers',
              'name': 'CPU %',
              'type': 'scatter',
              'uid': 'ce43a93c-0f5a-4304-8afa-01cf1aa45ff2',
              'x': [],
              'y': [],
              'yaxis': 'y2'},
             {'line': {'color': '#ff9800', 'dash': 'dot', 'width': 1.5},
              'marker': {'size': 4, 'symbol': 'square'},
              'mode': 'lines+markers',
              'name': 'Memory %',
              'type': 'scatter',
              'uid': '19c7d00a-85d6-4a29-8d70-76c230252560',
 